In [4]:
# ======================================================
# IMPORTING LIBRARIES
# ======================================================
import numpy as np
import pandas as pd
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [5]:
# =====================================
# LOADING DATASET
# =====================================

# Dataset has no headers so I am manually defining column names
cols = ['sentiment','id','date','query','user','text']

df = pd.read_csv(r"C:\Aishwary\Star Agile\Projects Dataset\Project3\training.1600000.processed.noemoticon.csv",
    encoding='latin-1',
    header=None,
    names=cols,
    nrows=200000  # limiting rows so training is faster on laptop
)

# Keeping only sentiment and tweet text
df = df[['sentiment','text']]

# Converting labels: 4(positive) -> 1
df['sentiment'] = df['sentiment'].replace(4,1)

print(df.head())

   sentiment                                               text
0          0  @switchfoot http://twitpic.com/2y1zl - Awww, t...
1          0  is upset that he can't update his Facebook by ...
2          0  @Kenichan I dived many times for the ball. Man...
3          0    my whole body feels itchy and like its on fire 
4          0  @nationwideclass no, it's not behaving at all....


In [6]:
# ===========================================
# TEXT CLEANING
# ===========================================

# Cleaning tweets helps model learn meaningful patterns
def clean_text(text):
    text = text.lower()
    text = re.sub(r'https\S+','',text)
    text = re.sub(r'@\w+','',text)
    text = re.sub(r'[^a-zA-Z\s]','',text)
    return text

df['text'] = df['text'].apply(clean_text)

In [7]:
# ===========================================
# TRAIN TEST SPLIT
# ===========================================

X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df['sentiment'],
    test_size=0.2,
    random_state=42
)

In [8]:
# =========================================
# BUILD VOCABULARY
# =========================================

# Neural networks need numbers instead of words
# So I am creating a simple word-to-index dictionary

vocab = {}
index = 1  # starting from 1 (0 will be padding)

for sentence in X_train:
    for word in sentence.split():
        if word not in vocab:
            vocab[word] = index
            index += 1

print("Vocabulary Size:", len(vocab))

Vocabulary Size: 86933


In [9]:
# ===========================================
# CONVERT TEXT TO SEQUENCES
# ===========================================

max_len = 30 # maximum tweet length

def encode(sentence):
    # converting words into their numeric index
    seq = [vocab.get(word,0) for word in sentence.split()]

    # padding or trimmimg sentence length
    if len(seq) < max_len:
        seq = seq+[0]*(max_len-len(seq))
    else:
        seq = seq[:max_len]
    return seq 

X_train_seq = [encode(s) for s in X_train]
X_test_seq = [encode(s) for s in X_test]

In [10]:
# ==================================================
# CREATING PYTORCH DATASET
# ==================================================

class TweetDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X,dtype=torch.long)
        self.y = torch.tensor(y.values,dtype=torch.float32)

    def __len__(self):
            return len(self.X)

    def __getitem__(self, idx):
            return self.X[idx], self.y[idx]

train_dataset = TweetDataset(X_train_seq, y_train)
test_dataset = TweetDataset(X_test_seq, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [11]:
# =====================================================
# BUILDING LSTM MODEL (PYTORCH)
# =====================================================

class SentimentModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()

        #Embedding layer learns words representations
        self.embedding = nn.Embedding (vocab_size, embed_dim)

        #LSTM learns sequence patterns
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

        # Fully connected output layer
        self.fc = nn.Linear(hidden_dim, 1)

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        out = self.fc(hidden[-1])
        out = self.sigmoid(out)
        return out.squeeze(1)

# Initializing model
vocab_size = len(vocab) + 1
model = SentimentModel(vocab_size, embed_dim=64, hidden_dim=32)

print(model)

SentimentModel(
  (embedding): Embedding(86934, 64)
  (lstm): LSTM(64, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [12]:
# =======================================
# TRAINING SETUP
# =======================================


criterion = nn.BCELoss()    # Binary classification loss
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [13]:
# ==============================================
# TRAINING LOOP
# ==============================================

epochs = 3

for epoch in range(epochs):
    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion (outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss+= loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 10.3115
Epoch 2, Loss: 0.0706
Epoch 3, Loss: 0.0161


In [14]:
# =================================================
# EVALUATION
# =================================================

correct = 0
total = 0

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)
        preds = (outputs > 0.5).float()
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

print ("Test Accuracy:", correct/total)

Test Accuracy: 1.0


In [15]:
# ==========================================
# PREDICTION FUNCTION
# ==========================================

def predict_sentiment(text):
    text = clean_text(text)
    seq = encode(text)
    tensor = torch.tensor([seq], dtype=torch.long)

    with torch.no_grad():
        pred = model(tensor).item()

        return "Positive" if pred > 0.5 else "Negative"

print(predict_sentiment("This product is amazing"))
print(predict_sentiment("Worst purchase ever"))

Negative
Negative
